In [1]:
# Isabella Alvarez
# Homework 10
# November 6, 2026 

In [2]:
import numpy as np
import import_ipynb
from hw9_ialvarez27 import *
from P7_ialvarez27.P6_ialvarez27_support_functions import *

Data Directory: hw6_data_export/
FITS extension: .fits
Last element of objfile: rdpharocor_stars_13s_9
Last element of darkfile: rdpharocor_dark_13s_5
Data Array size: 1024 , 1024
ny: 1024
nx: 1024
Number of files in objfile: 9
NUmber of files in darkfile: 5
Object cube shape: (9, 1024, 1024)
Dark cube shape:   (5, 1024, 1024)
Object observation date: 2002-01-27
Dark observation date: 2002-01-27
Problem 2a): the function is np.median()
Problem 2b) 
 Pixel value at [217,184]: 22.0
Problem 2c): Adding a History Entry
Problem 2d): Writing modified code to a file
Problem 2e):
Value of pixel index before dark subtraction: 10879.0
Value of pixel index after dark subtraction: 10857.0
Problem 2) Writing the function
Problem 2 continued
Value of pixel [217,184] of the last frame is -97.3964
Value at pixel [14,14]:   True
Value at pixel [2,1]  :   False
5
5
 Value of [217, 184]: 1.0278
[-118.73921889   35.91999427   10.15276388  146.13113485   63.27639047
  -53.01343684    0.         -110.918199

In [3]:
# Code from Practicum 7
def apphot(image, mask, radius, sky_radii, center):
    """
    Performs aperture photomtery on a target object.

    Parameters:
    -----------
    image: 2D numpy_like array
         The full astronomical image containing brightness pixel values
    mask: 2D numpy_like array
         Boolean array where True (1) represents good pixels and False (0) 
         represents bad pixels
    radius: float
        Radius of the star's photomtery aperture
    sky_radii: tuple of (float,float)
              Inner and outer sky annulus radii
    center: tuple of (float, float)
            Star's center coordinates (row, col)

    Returns:
    --------
    tuple: (stellar_flux, avg_sky, [optional] bad_pixels)
    """
    # Assigning to variables
    x , y = center 
    r_in, r_out = sky_radii

    # Creating sub-image boundaries 
    r = int(np.ceil(r_out))
    
    x_min = int(x) - r
    x_max = int(x) + r + 1
   
    y_min = int(y) - r
    y_max = int(y) + r +1

    # Extracting sky-image 
    sub_image = image[y_min : y_max, x_min :x_max]
    sub_mask  = mask [y_min : y_max, x_min :x_max]

    # Center coordinates relative to sub-image 
    sub_center = (y - y_min, x - x_min) # FIXED
    
    # Circular masks
    inner_disk = disk(r_in,  sub_center, sub_image.shape)
    outer_disk = disk(r_out, sub_center, sub_image.shape)

    # Selecting pixels in sky annulus 
    sky_annulus = outer_disk & ~inner_disk

    # Exclude bad pixels 
    good_sky = sky_annulus & sub_mask 

    # Average sky brightness
    avg_sky = np.mean(sub_image[good_sky])

    # Subtract background from the sub image
    sky_subtracted = sub_image - avg_sky

    # Making a mask for the photometry aperture
    aperture      = disk(radius, sub_center, sub_image.shape)
    good_aperture = aperture & sub_mask

    # Calculating total stellar flux 
    stellar_flux = np.sum(sky_subtracted[good_aperture])

    return stellar_flux, avg_sky

# Creating 21 x 21 image 
size  = 21
x0    = 10
y0    = 10
sigma = 2

# Generating coordinate arrays
y, x = np.indices((size, size))

# Creating a noiseless 2D Gaussian star
star = np.exp(-((x - x0)**2 + (y- y0)**2) / (2*sigma**2) )

# Mask where every pixel is good 
mask = np.ones((21, 21), dtype = bool)

# Run aperture photometry 
flux, sky = apphot(star, mask, 6, (7,10),(10,10))

print(f"Stellar Flux: {flux: .4f}")
print(f"Average Sky:  {sky: .4f}")

Stellar Flux:  24.8035
Average Sky:   0.0004


In [8]:
print("Problem 2: Writing the function")

def dophot(table, cube, mask, radius, sky_radii):
    """
    Performs aperture photometry for each star in each frame. 

    Parameters:
    -----------
    table: astropy.table 
          Table containing target coordindates.
    cube: 3D numpy array_like
           3D array of the observational frames. 
    mask: numpy array_like
        An array identifying good and bad pixels.
    radius: float
         Radius of the photomtery aperture. 
    sky_radii: tuple of (float,float)
              Inner and outer sky annulus radii

    Returns:
    --------
    table: astropy.table
           Targeted coordinates that went through aperture photometry
    """
    # Make a copy of the original table 
    result = table.copy()
    
    # Loop through each star 
    for i in range(result.shape[0]):
        
        # Loop through each observational frame
        for j in range(result.shape[1]):
            y      = table [i, j, 3]
            x      = table [i, j, 4] 
            center = (x , y )       # Updated star center
            image  = cube[j, : , :] # Select correct 2D image

            # Peform aperture photometry
            stellar_flux, avg_sky = apphot(image, mask, radius,
                                           sky_radii, center)
            # Update photometry table 
            result[i, j, 5] = stellar_flux
            result[i, j, 6] = avg_sky
    return result
            

Problem 2: Writing the function


In [7]:
print("Problem 3 Continued: ")

# Mask matching observational images
mask = np.all(np.isfinite(no_sky), axis=0)

# Using aperture and sky annulus radii values obtain from hw9
photometry_results = dophot(photometry, no_sky, mask, aperture_radius,(sky_inner, sky_outer))
print(photometry_results)

# Reshape pohotometry table into 2D
results_2d = photometry_results.reshape(9,7)

# Save results to an ASCII text file
np.savetxt("hw10_ialvarez27_problem2_photometry.txt", results_2d, fmt = "%.6f", 
           header = "yguess xguess width cy cx stellar_flux avg_sky")

Problem 3 Continued: 
[[[ 6.98000000e+02  5.12000000e+02  1.00573550e+00  6.98576131e+02
    5.12376727e+02  1.66727783e+04  4.45431535e+00]
  [ 7.21000000e+02  5.66000000e+02  4.10165737e-01  7.20701202e+02
    5.65661621e+02  1.39792620e+03  6.19391831e+00]
  [ 7.21000000e+02  5.66000000e+02  3.59768908e-01  7.20573042e+02
    5.65998852e+02  1.34807930e+03  1.80138245e+01]]

 [[ 6.68000000e+02  5.20000000e+02  1.06026082e+00  6.67872643e+02
    5.20249465e+02  3.84451318e+04  5.51409670e+01]
  [ 6.91000000e+02  5.74000000e+02  1.25918724e+00  6.90786974e+02
    5.75998893e+02 -1.18966156e+03  2.60156693e+01]
  [ 6.91000000e+02  5.74000000e+02  7.40447534e-01  6.91393213e+02
    5.72890406e+02  1.51712028e+02  1.41367528e+01]]

 [[ 5.68000000e+02  2.83000000e+02  1.05092378e+00  5.68265271e+02
    2.83301751e+02  3.18872055e+04  6.80064249e+01]
  [ 5.91000000e+02  3.37000000e+02  7.65813110e-01  5.90516520e+02
    3.35615762e+02 -2.93837972e+02  2.02233964e+00]
  [ 5.91000000e+02  3.